## Functions

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import sys, os
from ipywidgets import interact
from IPython.display import display, Math
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display, Math, Latex
from scipy.special import genlaguerre, legendre, lpmv, hermite
from matplotlib import cm
from scipy.special import sph_harm, genlaguerre, factorial, eval_hermite
import matplotlib as mpl
%matplotlib inline

In [3]:
hbar = 6.626e-34/(2*np.pi)
h = 6.626e-34
c = 3e10 #cm/s
mp = 1.67e-27
def Nv(v, k, mu):
    alpha = (k*mu/(hbar**2))**.5
    den = (2** v * factorial(v))**.5
    return (alpha/np.pi)**.25 / den

def psiv(x, v, k, mu):
    norm = Nv(v, k, mu)
    alpha = (k*mu/(hbar**2))**.5
    return norm *eval_hermite(v, x*alpha**.5) *np.exp(-alpha * x**2/2)

def Ev(v, k, mu):
    return (k/mu)**.5 * (v + .5) * hbar

def x0v(k, mu):
    omega  = (k/mu)**.5
    return (hbar/(mu*omega))**.5

def Vmorse(x, De, k, mu):
    beta = (k / (2*(De)))**.5
    Re = x0v(k, mu)
    return De * ((1 - np.exp(-beta * (x - Re)))**2 - 1)

def Emorse(v, k, mu, De):
    hw = hbar * np.sqrt(k / mu)
    return hw*(v+0.5) - hw**2*(v+0.5)**2/(4*De)

In [7]:
elements = {
    "H":  {"Z": 1,  "n": 1, "mass": 1.0078},   # 1H
    "D":  {"Z": 1,  "n": 1, "mass": 2.0078},   # 1H

    "Li": {"Z": 3,  "n": 2, "mass": 7.0160},   # 7Li
    "Be": {"Z": 4,  "n": 2, "mass": 9.0122},   # 9Be
    "B":  {"Z": 5,  "n": 2, "mass": 11.0093},  # 11B
    "C":  {"Z": 6,  "n": 2, "mass": 12.0000},  # 12C
    "N":  {"Z": 7,  "n": 2, "mass": 14.0031},  # 14N
    "O":  {"Z": 8,  "n": 2, "mass": 15.9949},  # 16O
    "F":  {"Z": 9,  "n": 2, "mass": 18.9984},  # 19F

    "Na": {"Z": 11, "n": 3, "mass": 22.9898},  # 23Na
    "Si": {"Z": 14, "n": 3, "mass": 27.9769},  # 28Si
    "P":  {"Z": 15, "n": 3, "mass": 30.9738},  # 31P
    "S":  {"Z": 16, "n": 3, "mass": 31.9721},  # 32S
    "Cl": {"Z": 17, "n": 3, "mass": 34.9689},  # 35Cl

    "K":  {"Z": 19, "n": 4, "mass": 38.9637},  # 39K
    "Ca": {"Z": 20, "n": 4, "mass": 39.9626},  # 40Ca
    "As": {"Z": 33, "n": 4, "mass": 74.9216},  # 75As
    "Se": {"Z": 34, "n": 4, "mass": 79.9165},  # 80Se
    "Br": {"Z": 35, "n": 4, "mass": 78.9183},  # 79Br
    "I": {"Z": 53, "n": 5, "mass": 127},  # 79Br
}

In [13]:
def QAO():
    @interact(
    plot =widgets.Dropdown(options=['Wavefunction', 'Probability'], value='Wavefunction'),
    E_max=widgets.FloatSlider(min=3000, max=50000, value=50000),
    k = widgets.FloatSlider(min=15, max=2300, value=478),
    ElementA = widgets.Dropdown(options=elements.keys(), value='H'),
    ElementB = widgets.Dropdown(options=elements.keys(), value='Cl'),
    De = widgets.FloatSlider(min=1.5, max=50, value=4.4),
    v = widgets.Dropdown(options=range(10), value=0),
    )
    def g(**kargs):
        mA = elements[kargs['ElementA']]['mass']
        mB = elements[kargs['ElementB']]['mass']
        k = kargs['k']
        mu = mA * mB / (mA + mB)

        mukg = mu*mp

        nu0 = (k/mukg)**.5 / c / (2*np.pi)



        display(Latex(f'Vib. constant: $ \\tilde{{\\nu}} = {nu0:.2f}~~ \\mathrm{{cm}}^{{-1}}$'))
        x0 = x0v(k, mukg)
        #x = np.linspace(-7*x0, 7*x0, 300)
        x = np.linspace(-1e-10, 2e-10, 300)
        

        fig = plt.figure(figsize=(10,5))
        gs = GridSpec(1,2,fig, wspace=.4)
        axE = fig.add_subplot(gs[1])
        De = kargs['De']*1.609e-19
        Vm = Vmorse(x, De,k,mukg) - np.min( Vmorse(x, De,k,mukg))
        axE.plot((x)*1e10, k*x**2/2/(h*c), 'k--', alpha=.5)
        axE.plot((x-x0)*1e10, (Vm - np.min(Vm))/(h*c))
        for vE in range(20):
            E = Emorse(vE, k, mukg, De) 

            negi = np.where(x < x0)[0]    # better: split around equilibrium
            posi = np.where(x > x0)[0]

            i0 = negi[np.argmin(np.abs(Vm[negi] - E))]
            i1 = posi[np.argmin(np.abs(Vm[posi] - E))]
            
            #xt = (2 * E / k)**.5
            Ex = (np.array([x[i0],x[i1]])-x0)*1e10 
            axE.plot(Ex, np.ones_like(Ex)*Emorse(vE, k, mukg, De)/(h*c), c='k')
            if kargs['v']==vE:
                axE.plot(Ex, np.ones_like(Ex)*Emorse(vE, k, mukg, De)/(h*c), c='r')
                xavg = np.average(Ex)*1e-10
                xrange = abs(Ex[1]-Ex[0])
            
            E = Ev(vE, k, mukg)
            xt = (2 * E / k)**.5
            Ex = np.array([-1,1])*xt*1e10
            axE.plot(Ex, np.ones_like(Ex)*Ev(vE, k, mukg)/(h*c), c='k', alpha=.3)
            if kargs['v']==vE:
                xrange0 = abs(Ex[1]-Ex[0])
        
        ax = fig.add_subplot(gs[0])

        scale = xrange/xrange0

        if kargs['plot']=='Wavefunction':
            ax.plot((x)*1e10, psiv((x-xavg)/scale,kargs['v'],k,mukg)*1e-5)
            ax.set_ylabel(f'$\\psi_{{v={kargs['v']}}}$ (m$^{{-1/2}}$)')
        elif kargs['plot']=='Probability':
            ax.plot((x)*1e10, psiv(x-xavg,kargs['v'],k,mukg)**2*1e-10)
            ax.set_ylabel(f'|$\\psi_{{v={kargs['v']}}}|^2$ (m$^{{-1}}$)')
        ax.axhline(0, linestyle='--', c='k')
        ax.set_xlabel('$r-r_{eq}$ ($\\AA$)')
        axE.set_xlabel('$r-r_{eq}$ ($\\AA$)')
        axE.set_ylabel('$E$ (cm$^{-1}$)')
        v = kargs['v']
        ax.set_title(f'$v={v}$, $E_{{v={v}}}= {Ev(v, k, mukg)/(h*c):,.0f}$ cm$^{{-1}}$')
        
        axE.set_ylim(0, kargs['E_max'])
            


        plt.show()

## Quantum Anharmonic Oscillator

In [12]:
QAO()

interactive(children=(Dropdown(description='plot', options=('Wavefunction', 'Probability'), value='Wavefunctio…